# Bonus — How Machine Learning Models Are Tested

**Bonus Module · AI Testing course**

---

The nine modules of this course tested **LLMs and agents** — probabilistic, text-in/text-out systems judged with semantic metrics and LLM judges. But "AI testing" started long before LLMs, with **classical machine learning**: models that predict a number or a category from tabular data. Testing those is more concrete — the metrics are exact math, and the "held-out test set" is the original version of every idea you've used since.

## What we'll cover
1. What machine learning actually is (vs. traditional software)
2. The **types** of ML — and where regression & classification fit
3. **Regression** — predicting a number, and how it's scored (MAE, MSE, RMSE, R²)
4. **Classification** — predicting a category, and how it's scored (accuracy, precision, recall, F1, confusion matrix)
5. **How testing works in ML** — train/test split, overfitting, cross-validation, baselines
6. How all of this maps back to the testing mindset from Modules 3–4

> **Runs fully offline & free** — pure `scikit-learn` on built-in datasets, in this module's **own isolated `.venv`** (so its ML stack never mixes with the LLM-testing environment from Modules 1–9).

## 1. What is machine learning?

In **traditional software** (Module 3's framing), a human writes explicit rules: `if income > 50000 and age > 25: approve`. The logic is in the code; you can read it.

In **machine learning**, you *don't* write the rules. You show the computer many **examples** (data with known answers), and an algorithm **learns** a rule that maps inputs to outputs. The "logic" ends up as numbers (weights) fit to the data — not lines you can read.

> **Plain English:** traditional software is a recipe you write step by step. ML is handing a cook 10,000 dishes labelled "tasty / not tasty" and letting them infer the recipe. Powerful — but now you can't just *read* the code to know it's correct. **You have to test it on examples it never saw.** That single fact is the root of everything below (and of LLM testing too).

## 2. Types of machine learning

| Type | What it learns from | Goal | Everyday example |
|---|---|---|---|
| **Supervised** | Labelled examples (input → known answer) | Predict the answer for new inputs | Spam filter, price prediction |
| **Unsupervised** | Unlabelled data (no answers) | Find structure / groups | Customer segmentation, anomaly detection |
| **Reinforcement** | Rewards from trial and error | Learn a policy of actions | Game-playing, robotics |

Most ML you'll test is **supervised**, and it splits into two by *what kind of answer* it predicts:

- **Regression** → predict a **number** on a continuous scale (house price, temperature, delivery time).
- **Classification** → predict a **category** from a fixed set (spam / not-spam, disease / healthy, digit 0–9).

Different answer types need different **metrics** — that's the heart of ML testing, and what we'll build next.

In [1]:
# Setup — all offline, no downloads, no keys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)

RANDOM_STATE = 42   # fixing the seed makes the split (and results) reproducible — itself a testing practice
print("ready")

ready


## 3. Regression — predicting a number

We'll use scikit-learn's built-in **diabetes** dataset: 10 health measurements per patient (age, BMI, blood pressure, …) and a number measuring disease progression a year later. The task: **predict that number** from the measurements.

The golden rule: **train on one slice of the data, test on a slice the model never saw.** If you score a model on data it trained on, it can just memorise — you'd be marking its own homework.

In [2]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression

X, y = load_diabetes(return_X_y=True)
print("dataset:", X.shape[0], "patients,", X.shape[1], "features")

# Hold out 20% as an unseen test set — the model is graded only on this.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

model = LinearRegression().fit(X_train, y_train)   # "learn" the mapping from training data
preds = model.predict(X_test)                      # predict on unseen patients

# Peek at a few predictions vs. the truth
for p, actual in list(zip(preds, y_test))[:5]:
    print(f"predicted {p:6.1f}   |   actual {actual:6.1f}   |   off by {abs(p-actual):5.1f}")

dataset: 442 patients, 10 features
predicted  139.5   |   actual  219.0   |   off by  79.5
predicted  179.5   |   actual   70.0   |   off by 109.5
predicted  134.0   |   actual  202.0   |   off by  68.0
predicted  291.4   |   actual  230.0   |   off by  61.4
predicted  123.8   |   actual  111.0   |   off by  12.8


### How regression is scored

A regression prediction is rarely *exactly* right, so we measure **how far off** it is:

| Metric | Plain English | Lower/Higher better |
|---|---|---|
| **MAE** (Mean Absolute Error) | average size of the miss, in the target's units | lower = better |
| **MSE** (Mean Squared Error) | average of squared misses — punishes big misses much harder | lower = better |
| **RMSE** (Root MSE) | MSE back in the target's units (√MSE) — the most common headline number | lower = better |
| **R²** (R-squared) | fraction of the variation the model explains: 1.0 = perfect, 0.0 = no better than guessing the mean, <0 = worse than the mean | higher = better |

A metric alone is meaningless without a **baseline**. The simplest baseline: always predict the training mean. A real model must beat that.

In [3]:
from sklearn.dummy import DummyRegressor

mae  = mean_absolute_error(y_test, preds)
mse  = mean_squared_error(y_test, preds)
rmse = mse ** 0.5
r2   = r2_score(y_test, preds)

print(f"MAE  : {mae:6.1f}   (average miss)")
print(f"MSE  : {mse:6.1f}")
print(f"RMSE : {rmse:6.1f}   (typical miss, in target units)")
print(f"R2   : {r2:6.3f}   (1.0 = perfect, 0.0 = no better than the mean)")

# Baseline: always predict the mean. Does our model actually beat it?
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
base_r2 = r2_score(y_test, baseline.predict(X_test))
print(f"\nbaseline R2 (predict the mean): {base_r2:.3f}")
print("model beats baseline:", r2 > base_r2)

MAE  :   42.8   (average miss)
MSE  : 2900.2
RMSE :   53.9   (typical miss, in target units)
R2   :  0.453   (1.0 = perfect, 0.0 = no better than the mean)

baseline R2 (predict the mean): -0.012
model beats baseline: True


## 4. Classification — predicting a category

Now a **category**, not a number. Scikit-learn's built-in **breast cancer** dataset: 30 measurements from a tumour scan, and a label — **malignant (0)** or **benign (1)**. The task: predict the label.

Here a plain "accuracy" number can *lie*, which is why classification needs several metrics — as you'll see.

In [4]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression

data = load_breast_cancer()
Xc, yc = data.data, data.target
print("classes:", dict(zip(range(len(data.target_names)), data.target_names)))
print("dataset:", Xc.shape[0], "tumours,", Xc.shape[1], "features")

Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.2, random_state=RANDOM_STATE, stratify=yc)
clf = LogisticRegression(max_iter=5000).fit(Xc_tr, yc_tr)
yc_pred = clf.predict(Xc_te)
print("\nfirst 10 predictions:", yc_pred[:10])
print("first 10 truths     :", yc_te[:10])

classes: {0: np.str_('malignant'), 1: np.str_('benign')}
dataset: 569 tumours, 30 features

first 10 predictions: [0 1 0 1 0 1 1 0 0 0]
first 10 truths     : [0 1 0 1 0 1 1 0 0 0]


### How classification is scored

Start with the **confusion matrix** — every prediction falls into one of four boxes (taking "malignant" as the case we most want to catch — a **positive**):

|  | predicted malignant | predicted benign |
|---|---|---|
| **actually malignant** | ✅ True Positive (TP) | ❌ False Negative (FN) — *missed a cancer* |
| **actually benign** | ❌ False Positive (FP) — *false alarm* | ✅ True Negative (TN) |

From those four boxes come the metrics:

| Metric | Formula (plain English) | Answers |
|---|---|---|
| **Accuracy** | (TP+TN) / everything | "what fraction did I get right?" |
| **Precision** | TP / (TP+FP) | "when I said malignant, how often was I right?" |
| **Recall** | TP / (TP+FN) | "of all real malignancies, how many did I catch?" |
| **F1** | harmonic mean of precision & recall | one number balancing the two |

**Why accuracy lies:** if 99% of samples are benign, a model that *always says benign* scores 99% accuracy — while catching **zero** cancers (recall = 0). On imbalanced or high-stakes problems, **recall and precision matter more than accuracy.**

In [5]:
from sklearn.dummy import DummyClassifier

# For a cancer screen, "malignant" (label 0) is the case we must not miss -> pos_label=0
acc  = accuracy_score(yc_te, yc_pred)
prec = precision_score(yc_te, yc_pred, pos_label=0)
rec  = recall_score(yc_te, yc_pred, pos_label=0)
f1   = f1_score(yc_te, yc_pred, pos_label=0)

print(f"Accuracy : {acc:.3f}")
print(f"Precision: {prec:.3f}   (of predicted-malignant, how many really were)")
print(f"Recall   : {rec:.3f}   (of real malignancies, how many we caught)  <- the critical one here")
print(f"F1       : {f1:.3f}")

print("\nConfusion matrix [rows=actual, cols=predicted], labels [malignant, benign]:")
print(confusion_matrix(yc_te, yc_pred, labels=[0, 1]))

# The 'always predict the majority class' trap:
dummy = DummyClassifier(strategy="most_frequent").fit(Xc_tr, yc_tr)
dummy_pred = dummy.predict(Xc_te)
print(f"\nDummy 'always benign' -> accuracy {accuracy_score(yc_te, dummy_pred):.3f}, "
      f"but malignant recall {recall_score(yc_te, dummy_pred, pos_label=0):.3f}  (catches no cancer!)")

Accuracy : 0.965
Precision: 0.975   (of predicted-malignant, how many really were)
Recall   : 0.929   (of real malignancies, how many we caught)  <- the critical one here
F1       : 0.951

Confusion matrix [rows=actual, cols=predicted], labels [malignant, benign]:
[[39  3]
 [ 1 71]]

Dummy 'always benign' -> accuracy 0.632, but malignant recall 0.000  (catches no cancer!)


In [6]:
# The confusion matrix is easier to read as a picture
import matplotlib
matplotlib.use("Agg")  # notebook will still render inline; Agg keeps it headless-safe
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay.from_predictions(
    yc_te, yc_pred, display_labels=data.target_names, cmap="Blues", ax=ax, colorbar=False
)
ax.set_title("Breast cancer — confusion matrix")
plt.tight_layout()
plt.show()
print("Off-diagonal cells are the mistakes. The top-right (missed malignancies) is the scary one.")

ModuleNotFoundError: No module named 'matplotlib'

## 5. How testing actually works in ML

Metrics tell you *how good*; the **testing discipline** is about making that number **trustworthy**. The essentials:

### a) Train / validation / test split
- **Train** — the model learns here.
- **Validation** — you tune choices here (which model, which settings).
- **Test** — touched **once**, at the end, to estimate real-world performance. If you peek at the test set while tuning, you've leaked it, and your number is a lie.

### b) Overfitting vs. underfitting
- **Overfitting** — the model *memorises* the training data (great train score, poor test score). The ML version of "it passed because it had seen the answers."
- **Underfitting** — too simple to capture the pattern (poor on both).
The gap between **train score and test score** is your overfitting alarm.

In [7]:
from sklearn.tree import DecisionTreeClassifier

# An unrestricted decision tree can memorise the training set perfectly.
overfit = DecisionTreeClassifier(random_state=RANDOM_STATE).fit(Xc_tr, yc_tr)
print("UNRESTRICTED tree")
print(f"  train accuracy: {overfit.score(Xc_tr, yc_tr):.3f}   test accuracy: {overfit.score(Xc_te, yc_te):.3f}")
print("  -> near-perfect on train, worse on test = OVERFITTING (memorised, didn't generalise)")

# Constrain it (limit depth) so it must learn a simpler, general rule.
better = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE).fit(Xc_tr, yc_tr)
print("\nDEPTH-LIMITED tree")
print(f"  train accuracy: {better.score(Xc_tr, yc_tr):.3f}   test accuracy: {better.score(Xc_te, yc_te):.3f}")
print("  -> smaller train/test gap = generalises better")

UNRESTRICTED tree
  train accuracy: 1.000   test accuracy: 0.912
  -> near-perfect on train, worse on test = OVERFITTING (memorised, didn't generalise)

DEPTH-LIMITED tree
  train accuracy: 0.976   test accuracy: 0.939
  -> smaller train/test gap = generalises better


### c) Cross-validation — is the score just luck?

A single train/test split might get a lucky (or unlucky) slice. **k-fold cross-validation** splits the data k ways, trains/tests k times, and reports the spread. A trustworthy result is not just high — it's **stable** across folds.

In [8]:
scores = cross_val_score(LogisticRegression(max_iter=5000), Xc, yc, cv=5, scoring="accuracy")
print("5-fold accuracies:", np.round(scores, 3))
print(f"mean {scores.mean():.3f}  ±  {scores.std():.3f}")
print("A small spread means the score is reliable, not a fluke of one split.")

5-fold accuracies: [0.939 0.947 0.982 0.93  0.956]
mean 0.951  ±  0.018
A small spread means the score is reliable, not a fluke of one split.


### d) Two more ways ML tests silently lie
- **Data leakage** — information from the test set (or the future) sneaks into training (e.g. scaling using the whole dataset before splitting). The score looks great and collapses in production. Fix: split *first*, then fit any preprocessing on the training data only.
- **Distribution shift** — the model is tested on data like training, but the real world drifts (new users, new seasons). This is the classical-ML sibling of the **model drift** you met in Module 3 — the reason evaluation is continuous, not one-and-done.

## 6. It's the same testing mindset

Everything you learned for LLMs started here:

| Classical ML | …became, for LLMs/agents (this course) |
|---|---|
| Held-out **test set** the model never saw | Golden dataset run before every change (Module 4) |
| **Metrics** as the pass/fail signal (RMSE, F1) | Faithfulness, relevancy, GEval scores (Modules 4–9) |
| Evaluate on **data slices** (per group/segment) | Equivalence partitioning over inputs (Module 4 Day 4) |
| **Minority / hard cases** matter more than accuracy | Hard negatives; recall on the case that matters (Module 4 Day 4) |
| **Overfitting** = memorised, fails on new data | An agent that only works on your demo prompts |
| **Cross-validation** = don't trust one lucky run | Run the suite repeatedly; watch the spread |
| **Data leakage / drift** | Test-set contamination; model drift (Module 3) |

The tools change (RMSE → an LLM judge); the discipline is identical: **hold out data the system hasn't seen, measure with the right metric, mind the cases that matter, and never trust a single number.**

## Summary
- ML **learns rules from examples**, so you must test it on **unseen data**.
- **Regression** predicts numbers (MAE/MSE/RMSE/R²); **classification** predicts categories (accuracy/precision/recall/F1 + confusion matrix).
- **Accuracy can lie** — precision and recall tell you *which* mistakes you're making.
- Real ML testing = **train/test discipline, overfitting checks, cross-validation, baselines, and watching for leakage/drift** — the exact mindset this whole course applied to LLMs.